# SMS Spam Detection — Model Training and Evaluation

This notebook trains and evaluates an SMS spam classifier using the tab-separated dataset file `spam.tsv`. Place the dataset in the same folder as this notebook before running.

Expected fields: `label` (ham/spam) and `message` (SMS text). Extra dataset columns are ignored.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
import pickle


In [ ]:
# Load tab-separated dataset
df = pd.read_csv('spam.tsv', sep='\t')
df.head()

In [ ]:
# Keep relevant fields and clean data
df = df[['label', 'message']].dropna()
df['label'] = df['label'].astype(str).str.strip().str.lower()
df['message'] = df['message'].astype(str).str.strip()
df = df[df['label'].isin(['ham', 'spam'])]
df = df[df['message'] != '']
df['label'].value_counts()

In [ ]:
# Explore class distribution
df['label'].value_counts().plot(kind='bar', title='SMS Label Distribution')
plt.xlabel('Label')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['message'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', ngram_range=(1, 2))),
    ('classifier', MultinomialNB())
])
pipeline.fit(X_train, y_train)

In [ ]:
predictions = pipeline.predict(X_test)
print('Accuracy:', accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions, labels=['ham', 'spam']))
ConfusionMatrixDisplay.from_predictions(y_test, predictions, labels=['ham', 'spam'])
plt.title('SMS Spam Classifier — Confusion Matrix')
plt.show()

In [ ]:
# Save the full pipeline for the app
with open('model.pkl', 'wb') as file:
    pickle.dump(pipeline, file)
print('Saved full preprocessing + classifier pipeline to model.pkl')

In [ ]:
pipeline.predict(['Congratulations! You won a prize. Click the link now.'])

## Notes
- Place `spam.tsv` beside this notebook and run all cells.
- Run all cells to create a fresh `model.pkl` pipeline.
- Only load pickle files from trusted sources.
- Report test metrics from your own run; do not invent or copy metrics.